# Food Calorie Estimation Using Image Classification

**Objective:** classify a food image and estimate calories using a food-category lookup table and a user-entered portion weight.

> **Important limitation:** an image classifier identifies a food category; it does not measure food mass or reliably infer recipe, ingredients, or portion size. Calories below are approximate estimates based on the entered weight and editable reference values (kcal per 100 g).

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os

from tensorflow.keras.preprocessing.image import ImageDataGenerator

## 1. Configure the dataset

Arrange your images in folders named after each food class. For example:

```text
Food_Dataset/
    Apple/
    Banana/
    Burger/
    Ice_Cream/
    Pizza/
```

Each class folder should contain images of that food. The notebook will split images into training and validation sets automatically.

Set `DATA_DIR` to the full path of your dataset folder. On Windows, use a raw string such as `r"C:\Users\YourName\Desktop\Food_Dataset"`.

In [ ]:
from pathlib import Path

DATA_DIR = Path("Food_Dataset")  # Change this to your dataset folder path
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
VALIDATION_SPLIT = 0.2
SEED = 42

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Dataset folder not found: {DATA_DIR.resolve()}\n"
        "Update DATA_DIR in this cell to the folder containing one subfolder per food class."
    )

class_folders = sorted([p.name for p in DATA_DIR.iterdir() if p.is_dir()])
print("Detected food classes:", class_folders)

if len(class_folders) < 2:
    raise ValueError("Please provide at least two class folders, each containing food images.")

## 2. Load and split the image dataset

In [ ]:
train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,
    validation_split=VALIDATION_SPLIT,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True
)

validation_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,
    validation_split=VALIDATION_SPLIT
)

train_data = train_datagen.flow_from_directory(
    str(DATA_DIR),
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training",
    seed=SEED,
    shuffle=True
)

validation_data = validation_datagen.flow_from_directory(
    str(DATA_DIR),
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation",
    seed=SEED,
    shuffle=False
)

class_names = list(train_data.class_indices.keys())
num_classes = len(class_names)
print("Class indices:", train_data.class_indices)

## 3. Display sample images

In [ ]:
images, labels = next(train_data)
plt.figure(figsize=(12, 8))
for i in range(min(9, len(images))):
    plt.subplot(3, 3, i + 1)
    plt.imshow(images[i])
    plt.title(class_names[np.argmax(labels[i])])
    plt.axis("off")
plt.tight_layout()
plt.show()

## 4. Build the CNN image-classification model

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3)),
    tf.keras.layers.Conv2D(32, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Conv2D(64, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Conv2D(128, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

## 5. Train the model

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    )
]

EPOCHS = 15
history = model.fit(
    train_data,
    validation_data=validation_data,
    epochs=EPOCHS,
    callbacks=callbacks
)

## 6. Plot training and validation performance

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Training accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.title("Model Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.title("Model Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
plt.show()

## 7. Evaluate on the validation set

In [ ]:
val_loss, val_accuracy = model.evaluate(validation_data, verbose=1)
print(f"Validation loss: {val_loss:.4f}")
print(f"Validation accuracy: {val_accuracy * 100:.2f}%")

## 8. Classification report and confusion matrix

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

validation_data.reset()
probabilities = model.predict(validation_data, verbose=1)
y_pred = np.argmax(probabilities, axis=1)
y_true = validation_data.classes

print(classification_report(
    y_true,
    y_pred,
    labels=list(range(len(class_names))),
    target_names=class_names,
    zero_division=0
))

cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
fig, ax = plt.subplots(figsize=(max(7, len(class_names)), max(6, len(class_names) * 0.8)))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names).plot(
    ax=ax, cmap="Blues", xticks_rotation=45, colorbar=False
)
plt.title("Food Classification Confusion Matrix")
plt.tight_layout()
plt.show()

## 9. Create an editable calorie reference table

Values below are illustrative approximate kcal per 100 g for common foods. Edit them to match your chosen food categories and a reliable nutrition source. Food preparation and recipes can change calorie values substantially.

In [ ]:
# Keys must match the folder/class names (case-insensitive).
# Values are approximate kcal per 100 grams.
calories_per_100g = {
    "apple": 52,
    "banana": 89,
    "burger": 295,
    "ice_cream": 207,
    "pizza": 266
}

print("Classifier classes:", class_names)
print("Add or edit calorie values so every class has a matching entry.")

## 10. Predict a food image and estimate calories

In [ ]:
from tensorflow.keras.utils import load_img, img_to_array

def predict_food_and_calories(image_path, portion_weight_g=None):
    image_path = Path(image_path)
    if not image_path.exists():
        raise FileNotFoundError(f"Image not found: {image_path.resolve()}")

    image = load_img(image_path, target_size=IMAGE_SIZE)
    image_array = img_to_array(image).astype("float32") / 255.0
    image_array = np.expand_dims(image_array, axis=0)

    probabilities = model.predict(image_array, verbose=0)[0]
    predicted_index = int(np.argmax(probabilities))
    predicted_food = class_names[predicted_index]
    confidence = float(probabilities[predicted_index]) * 100

    print(f"Predicted food: {predicted_food}")
    print(f"Model confidence: {confidence:.2f}%")

    reference_value = calories_per_100g.get(predicted_food.lower())
    if reference_value is None:
        print(f"No calorie reference found for '{predicted_food}'. Add it to calories_per_100g.")
    else:
        print(f"Approximate calories per 100 g: {reference_value} kcal")
        if portion_weight_g is not None:
            estimated_kcal = reference_value * float(portion_weight_g) / 100
            print(f"Entered portion weight: {portion_weight_g:g} g")
            print(f"Estimated calories: {estimated_kcal:.1f} kcal")

    plt.figure(figsize=(5, 5))
    plt.imshow(load_img(image_path))
    plt.title(f"{predicted_food} ({confidence:.1f}%)")
    plt.axis("off")
    plt.show()

    return predicted_food, confidence

# Replace the path and portion weight with your own test image and measured weight.
# Example:
# predict_food_and_calories("test_pizza.jpg", portion_weight_g=150)

## 11. Save the trained model and class labels

In [ ]:
model.save("food_classification_model.keras")

import json
with open("food_class_names.json", "w", encoding="utf-8") as file:
    json.dump(class_names, file, indent=2)

print("Saved model: food_classification_model.keras")
print("Saved class labels: food_class_names.json")

## 12. Conclusion

This project trains a CNN to classify food images into the categories represented in the supplied dataset. The predicted category is connected to an editable nutrition lookup table to estimate calories for a user-entered portion weight.

**Limitations:** the model's prediction can be wrong, and the calorie estimate is only as reliable as the category prediction, reference nutrition value, and portion weight. It does not infer portion weight from the image or account for recipe-specific ingredients.

**Future improvements:** use a larger and more diverse food dataset, evaluate on a separate held-out test set, add more food categories, and use verified nutrition data.